In [1]:
# --- Бібліотеки для даної роботи ---
try:
    import markdown, numpy, pandas, plotly, nbformat
    print("Бібліотеки вже встановлені. Пропускаємо інсталяцію.")
except ImportError:
    print("Встановлюємо бібліотеки...")
    %pip install markdown numpy pandas plotly nbformat

Бібліотеки вже встановлені. Пропускаємо інсталяцію.


<div align="center">
    <h2><b>Architecture Design Document (ADD)</b></h2>
    <h1>Інтеграція й оптимізація комунікації в складній розподіленій системі</h1>
    <h3>Платформа бронювання авіаквитків «2BeFlüge»</h3>
    <br>

**Метадані документа:**

| **Тип документа:** | High-Level Design (HLD) & Communication Strategy |
| :--- | :--- |
| **Статус:** | Draft / В розробці |
| **Версія архітектури:** | 1.0.0 |
| **Дата створення:** | Березень 2026 р. |

<br>

**Автор проекту:**

| **Ім'я:** | Олег Гаценко |
| :--- | :--- |
| **Роль:** | System Architect / Студент магістратури |
| **Організація:** | NeoVersity (Master's in AI & Machine Learning) |

</div>

---

### **Executive Summary (Резюме архітектури):**

Цей документ описує високорівневу архітектуру (HLD) та стратегію комунікації для платформи бронювання авіаквитків «2BeFlüge». Система спроектована для витримки екстремальних навантажень під час флеш-розпродажів (Flash Sales) та базується на мікросервісній архітектурі. Ключовим архітектурним рішенням є гібридна комунікація: використання **gRPC** для синхронного ядра (критичні до затримки транзакції) та подійно-орієнтованої моделі (Event-Driven) на базі **Apache Kafka** для асинхронних підсистем (сповіщення, аналітика, ML-рекомендації).

### **Глобальна архітектурна схема (Високорівнева):**

```mermaid
flowchart TD
    Client([Web / Mobile<br/>Клієнти 2BeFlüge])
    Gateway{{API Gateway / GraphQL}}

    Client -->|API запити| Gateway

    subgraph Core ["Синхронне&nbsp;ядро&nbsp;(Low&nbsp;Latency)"]
        Booking[Сервіс бронювання]
        Inventory[Сервіс інвентарю]
        Payment[Платіжний сервіс]
    end

    Gateway -->|GraphQL / REST| Booking
    Booking <-->|gRPC| Inventory
    Booking <-->|gRPC| Payment
    Gateway -->|REST| Payment

    Broker[[Apache Kafka Broker]]

    Booking -.->|Event: BookingCreated| Broker
    Payment -.->|Event: PaymentProcessed| Broker

    subgraph Async ["Асинхронні&nbsp;підсистеми&nbsp;(Event&nbsp;Driven)"]
        Notification[Сервіс сповіщень]
        Analytics[Сервіс аналітики]
        Recommendation[Сервіс ML рекомендацій]
    end

    Broker -.->|Pub/Sub| Notification
    Broker -.->|Pub/Sub| Analytics
    Broker -.->|Pub/Sub| Recommendation

    classDef sync fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000
    classDef async fill:#fff8e1,stroke:#ff8f00,stroke-width:2px,stroke-dasharray: 5 5,color:#000
    classDef broker fill:#eceff1,stroke:#37474f,stroke-width:3px,color:#000

    class Booking,Inventory,Payment sync
    class Notification,Analytics,Recommendation async
    class Broker broker

    style Core fill:#ffea00,stroke:#1565c0,stroke-width:1px
    style Async fill: #009dff,stroke:#ff8f00,stroke-width:1px
```

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **1. Проектування комунікації між мікросервісами**

У високонавантаженій системі бронювання авіаквитків **2BeFlüge** вибір протоколу взаємодії залежить від того, чи знаходиться сервіс на **Критичному шляху** (Critical Path) користувача. 

Ми застосовуємо гібридний підхід: **Синхронна комунікація (gRPC/REST)** використовується виключно там, де необхідна строга консистентність даних (запобігання подвійному бронюванню), а **Асинхронна (Event-Driven)** — для масштабування побічних бізнес-процесів і захисту від каскадних відмов під час флеш-розпродажів.

### 1.1. Матриця взаємодії компонентів (Communication Matrix)

| Відправник | Отримувач | Тип зв'язку | Технологія | Обґрунтування (Trade-offs) |
| :--- | :--- | :--- | :--- | :--- |
| **Клієнт** (Web/Mobile) | **API Gateway** | Синхронний | **GraphQL** | Мобільні клієнти часто мають нестабільне з'єднання. GraphQL усуває проблеми *Overfetching* (завантаження зайвих даних) та *Underfetching* (необхідність робити 5 запитів замість 1). Клієнт сам формує структуру відповіді (маршрут + ціна + наявність місць). |
| **API Gateway** | **Сервіс бронювання** | Синхронний | **REST API** | Використовуємо REST (JSON через HTTP) як стандартний, легко кешований протокол для зовнішньої маршрутизації. Це спрощує налаштування Rate Limiting та WAF на рівні Gateway. |
| **Сервіс бронювання** | **Сервіс інвентарю** | Синхронний | **gRPC** | **Критично низька затримка.** Перевірка і резервування місця в літаку вимагає жорсткої консистентності (ACID). Бінарний формат Protobuf швидший за JSON у 5-7 разів, а строгі контракти усувають помилки типізації між сервісами. |
| **Сервіс бронювання** | **Платіжний сервіс** | Синхронний | **gRPC** | Ініціалізація транзакції та холдування (блокування) коштів вимагає надійного RPC-виклику. gRPC підтримує вбудовані дедлайни (Timeouts) та механізми скасування, що критично для фінансових операцій. |
| **Будь-який сервіс** | **Сервіс сповіщень** | Асинхронний | **RabbitMQ** (Message Queue) | Користувач не повинен чекати, поки SMTP-сервер надішле email. Використовуємо патерн *Point-to-Point* (черга задач). RabbitMQ ідеально підходить для гарантованої доставки (At-least-once) та має вбудований механізм Dead-Letter Queues (DLQ) для листів, що не відправилися. |
| **Ядро** (Бронювання, Оплати) | **Аналітика та Рекомендації (ML)** | Асинхронний | **Apache Kafka** (Event Stream) | Патерн *Pub/Sub*. Сервіси ядра публікують події (напр., `FlightSearched`, `TicketPurchased`). Аналітика і ML вичитують їх незалежно у реальному часі. Kafka обрана через здатність витримувати мільйони подій за секунду та можливість "перемотувати" історію подій, якщо ML-сервіс тимчасово впаде. |

**Як підсумок:** обраний гібридний стек гарантує блискавичну швидкодію та транзакційну цілісність критичного ядра, повністю ізолюючи його від важких фонових завдань, що дозволяє системі гнучко та незалежно масштабуватися.

---

### 1.2. Аналіз компромісів та сценарій «Флеш-розпродаж» (Flash Sale)

Під час акцій навантаження на систему зростає експоненціально. Якщо всі комунікації залишити синхронними, Платіжний сервіс (або зовнішній банківський шлюз) стане вузьким місцем (Bottleneck), що призведе до вичерпання пулу з'єднань і каскадного падіння всієї платформи.

**Шляхи оптимізації комунікації під час піків:**
1. **Зворотний тиск (Backpressure):** Замість того, щоб чекати синхронної відповіді від банку, API Gateway приймає запит на оплату, миттєво повертає клієнту HTTP статус `202 Accepted` ("Опрацьовується") і скидає завдання у швидку чергу (Kafka).
2. **Асинхронна обробка:** Воркери Платіжного сервісу забирають повідомлення з черги з тією швидкістю, яку фізично може витримати банківський API. 
3. **Event-Driven UI:** Клієнтський додаток (через WebSockets або Server-Sent Events) отримує асинхронне сповіщення про успішну покупку щойно Платіжний сервіс завершить транзакцію і опублікує подію `PaymentSuccess`.

**Візуалізація асинхронного потоку оплати (Sequence Diagram):**

```mermaid
sequenceDiagram
    autonumber
    actor Client as Клієнт (UI)
    participant GW as API Gateway
    participant Broker as Kafka (Черга)
    participant Worker as Payment Worker
    participant Bank as Банківський Шлюз

    Note over Client, Bank: Флеш-розпродаж (Пікове навантаження)

    Client->>GW: POST /api/v1/payments (Запит на оплату)
    GW->>Broker: Публікація події [PaymentRequested]
    Broker-->>GW: ACK (Подію успішно збережено)
    
    %% Швидка відповідь клієнту без блокування
    GW-->>Client: HTTP 202 Accepted (Платіж в обробці)
    Note left of Client: UI показує спіннер<br/>"Обробляємо оплату..."

    %% Асинхронна обробка в бекграунді
    Broker->>Worker: Консьюмер вичитує подію з черги
    Worker->>Bank: Синхронний виклик (в межах лімітів банку)
    Bank-->>Worker: Відповідь (Успіх)

    Worker->>Broker: Публікація події [PaymentSuccess]
    Note right of Broker: Notification Service вичитує<br/>подію і пушить клієнту
```

**Математичне обґрунтування (Закон Літтла):**

Фундаментальною причиною для переходу на асинхронні черги є класична теорія масового обслуговування (Queueing Theory), зокрема Закон Літтла:

$$L = \lambda \cdot W$$

Де:
* $L$ — кількість одночасних запитів у системі (наш пул з'єднань на Gateway).
* $\lambda$ — швидкість надходження нових запитів від користувачів.
* $W$ — середній час обробки одного запиту (обмежений повільним API банку).

Під час флеш-розпродажу параметр $\lambda$ зростає експоненціально. Оскільки ми фізично не можемо зменшити $W$ (швидкість зовнішнього банку є константою), параметр $L$ неминуче проб'є стелю лімітів нашого API Gateway. **Єдиний математично вірний спосіб** уникнути відмови всієї системи — це розірвати синхронний ланцюг і перенести $L$ у високоємну буферну чергу.

---

### 1.3. Неочевидні аспекти та еволюція технологій (Replaceability)

Вибір гібридного стека комунікацій тягне за собою кілька неочевидних архітектурних наслідків:

1. **Вплив на клієнтські додатки (UI/UX):** Перехід на асинхронну обробку (особливо оплат) вимагає зміни парадигми на фронтенді. Замість простого HTTP-запиту з очікуванням відповіді, клієнтські додатки мають підтримувати Event-Driven підхід (наприклад, піднімати **WebSocket**-з'єднання або використовувати **Server-Sent Events**), щоб слухати сповіщення від системи, коли бекграунд-процес завершиться.
2. **Складність моніторингу (Observability):** У розподіленій системі, де запит стрибає з REST у gRPC, а потім у Kafka, класичне логування стає неефективним. Це вимагає обов'язкового впровадження систем **Distributed Tracing** (наприклад, Jaeger або OpenTelemetry). Кожному початковому запиту на API Gateway присвоюється унікальний `Correlation-ID`, який передається у заголовках (Headers) або метаданих усіх наступних gRPC-викликів та повідомлень Kafka.
3. **Можливість заміни технологій (Replaceability):**
   - Завдяки використанню **API Gateway**, зовнішні клієнти повністю ізольовані від внутрішньої топології. Якщо в майбутньому ми вирішимо замінити REST на gRPC між Booking Service та іншими новими модулями, публічні клієнти цього навіть не помітять.
   - Для брокерів повідомлень застосовується патерн **Ports and Adapters (Гексагональна архітектура)**. Бізнес-логіка мікросервісу нічого не знає про специфіку RabbitMQ чи Kafka. Вона працює через абстрактний інтерфейс `MessagePublisher`. Якщо пропускної здатності RabbitMQ для `Notification Service` стане недостатньо, інженери зможуть замінити його на топік Kafka, переписавши лише один клас-адаптер, без жодного втручання в логіку формування самих сповіщень.